# Gemma 4 26B A4B Vision — FLN Worksheet Question Splitter (Kaggle)

Extracts **every individual question** from a worksheet image using intelligent segmentation,  
then analyzes each crop separately with Gemma 4 26B A4B. Outputs a **per-question JSON**  
with complete visible content — text, objects, answer spaces, structure — WITHOUT answers.

---
**Setup**  
1. Settings → Accelerator → **GPU T4 x2**  
2. Add-ons → Secrets → Add **HF_TOKEN** (your Hugging Face token)  
3. Upload images via **Add Data** button (top right)  
4. Cell → Run All

---
**How it works:**  
1. Preprocess image (deskew, sharpen, CLAHE)  
2. Segment into individual question regions via projection profiles  
3. Filter headers/footers/noise, recursively split large regions  
4. Send EACH crop to Gemma with a comprehensive extract-everything prompt  
5. Collect per-question JSON into worksheet-level output

---
**Note:** First run downloads the GGUF model (~17 GB, ~15 min on T4).
Cached in `/root/gguf_cache/` (root partition has ~66 GB free).
Model is split across both T4 GPUs (32 GB VRAM total) via llama.cpp automatic multi-GPU.


In [ ]:
import shutil, subprocess, os

shutil.rmtree("/kaggle/working/", ignore_errors=True)
try:
    shutil.rmtree("/tmp/", ignore_errors=True)
    os.makedirs("/tmp/", exist_ok=True)
except: pass
subprocess.run(["pip", "cache", "purge"], capture_output=True)

total, used, free = shutil.disk_usage("/kaggle/")
print(f"Disk free: {free // (1024**3)} GB")


In [ ]:
# Cell 1: GPU check + Install deps
import os, shutil, time, json, re, base64, zipfile
from pathlib import Path
from IPython.display import display, Image as IPyImage

import torch
if not torch.cuda.is_available():
    raise SystemExit('NO GPU. Go to Settings > Accelerator > GPU, then Run all again.')
print(f"GPU count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

OUTPUT_DIR = "/kaggle/working/FLN_Results"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Outputs: {OUTPUT_DIR}")

!pip install opencv-python -q 2>&1 | tail -1
import cv2, numpy as np
print("Ready")


In [ ]:
# Cell 2: Hugging Face Login
from huggingface_hub import login

token = os.environ.get('HF_TOKEN')
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass

if not token:
    raise ValueError('HF_TOKEN not found. Set it in Add-ons > Secrets.')

login(token)
print("Logged in")


In [ ]:
# Cell 3: Load Gemma 4 26B A4B (MoE)
import subprocess, sys

torch_ver = torch.version.cuda or ''
print(f"Torch CUDA: {torch_ver}")

wheel_map = {"12.1": "cu121", "12.2": "cu122", "12.3": "cu123",
             "12.4": "cu124", "12.5": "cu125", "12.6": "cu126"}
cuda_wheel = wheel_map.get(torch_ver, 'cu124')
print(f"Using {cuda_wheel} wheel")

print("Installing llama-cpp-python...")
sys.stdout.flush()
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'llama-cpp-python',
    f'--extra-index-url', f'https://abetlen.github.io/llama-cpp-python/whl/{cuda_wheel}',
    '--force-reinstall', '--no-cache-dir', '-q'
])
print("llama-cpp-python installed")

from llama_cpp import Llama
from llama_cpp.llama_chat_format import Gemma4ChatHandler

CACHE_DIR = "/root/gguf_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

CHAT_HANDLER = Gemma4ChatHandler.from_pretrained(
    repo_id="unsloth/gemma-4-26B-A4B-it-GGUF",
    filename="mmproj-F16.gguf",
    local_dir=CACHE_DIR, verbose=False,
)
print("Chat handler OK")

start = time.time()
llm = Llama.from_pretrained(
    repo_id="unsloth/gemma-4-26B-A4B-it-GGUF",
    filename="gemma-4-26B-A4B-it-UD-Q5_K_XL.gguf",
    local_dir=CACHE_DIR,
    chat_handler=CHAT_HANDLER,
    n_gpu_layers=-1, n_ctx=8192,
    flash_attn=True, verbose=False,
)
t = (time.time()-start)/60
print(f"Model loaded in {t:.1f} min")


In [ ]:
# Cell 4: Image preprocessing

def preprocess(img):
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    result = img.copy()
    applied = []

    is_blurry = lap_var < 80
    is_small = min(h, w) < 800
    is_low_contrast = (float(gray.max()) - float(gray.min())) < 100

    edges = cv2.Canny(gray, 50, 150, apertureSize=3)
    lines = cv2.HoughLines(edges, 1, np.pi/180, 200)
    angle = 0.0
    if lines is not None:
        angles = []
        for line in lines:
            theta = line[0][1]
            deg = np.degrees(theta) - 90
            if abs(deg) < 30:
                angles.append(deg)
        if angles:
            angle = np.median(angles)
    is_skewed = abs(angle) > 3.0

    if is_skewed:
        M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
        result = cv2.warpAffine(result, M, (w, h),
                               flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)
        applied.append(f"deskew {angle:.1f} deg")
        h, w = result.shape[:2]

    if is_blurry:
        blurred = cv2.GaussianBlur(result, (0, 0), 3.0)
        sharp = cv2.addWeighted(result, 1.5, blurred, -0.5, 0)
        sharp = np.clip(sharp, 0, 255).astype(np.uint8)
        new_var = cv2.Laplacian(cv2.cvtColor(sharp, cv2.COLOR_RGB2GRAY), cv2.CV_64F).var()
        if new_var > lap_var:
            result = sharp
            applied.append("unsharp")

    if is_low_contrast or is_blurry:
        lab = cv2.cvtColor(result, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        l = clahe.apply(l)
        result = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB)
        applied.append("CLAHE")

    if is_small:
        scale = max(1.0, 800 / min(h, w))
        if scale > 1.1:
            result = cv2.resize(result, None, fx=scale, fy=scale,
                               interpolation=cv2.INTER_CUBIC)
            applied.append(f"upscale {scale:.1f}x")

    info = {"blurry": bool(is_blurry), "skewed": bool(is_skewed), "small": bool(is_small),
            "low_contrast": bool(is_low_contrast), "lap_var": float(round(lap_var, 1)),
            "angle": float(round(angle, 1)), "original_size": f"{int(img.shape[1])}x{int(img.shape[0])}",
            "applied": applied}
    return result, info


In [ ]:
# Cell 5: Question Segmentation + Per-Question Extraction

# --- Segmentation ---
def filter_question_regions(questions, img_h, img_w):
    filtered = []
    for q in questions:
        b = q["bbox"]
        top_r = b["y"] / img_h
        bot_r = (b["y"] + b["height"]) / img_h
        aspect = b["width"] / max(b["height"], 1)
        area_r = (b["width"] * b["height"]) / (img_w * img_h)

        is_header = top_r < 0.08 and (b["height"] < 80 or (aspect > 5 and b["height"] < 120))
        is_footer = bot_r > 0.88 and b["height"] < 60
        is_bottom = bot_r > 0.95 and b["height"] < 100
        is_small = area_r < 0.005 and (b["height"] < 40 or b["width"] < 80)
        is_noise = b["height"] < 30 or b["width"] < 30

        if not (is_header or is_footer or is_bottom or is_small or is_noise):
            filtered.append(q)
    return filtered

def merge_fragments(regions):
    merged = []
    i = 0
    while i < len(regions):
        cur = regions[i]
        cb = cur["bbox"]
        is_frag = cb["height"] < 50 or (cb["height"] < 70 and cb["width"] > 5 * cb["height"])

        if is_frag and i + 1 < len(regions):
            nxt = regions[i + 1]
            nb = nxt["bbox"]
            gap = nb["y"] - (cb["y"] + cb["height"])
            if gap < 45:
                ny = cb["y"]
                nh = (nb["y"] + nb["height"]) - ny
                nx = min(cb["x"], nb["x"])
                nw = max(cb["width"], nb["width"])
                merged.append({"idx": 0, "bbox": {"x": nx, "y": ny, "width": nw, "height": nh}, "crop": None})
                i += 2
                continue

        if is_frag and merged:
            prev = merged[-1]
            pb = prev["bbox"]
            gap = cb["y"] - (pb["y"] + pb["height"])
            if gap < 45:
                merged[-1]["bbox"] = {
                    "x": min(pb["x"], cb["x"]),
                    "y": pb["y"],
                    "width": max(pb["width"], cb["width"]),
                    "height": (cb["y"] + cb["height"]) - pb["y"],
                }
                i += 1
                continue

        merged.append(cur)
        i += 1
    return merged

def deduplicate_overlaps(regions):
    kept = []
    for r in sorted(regions, key=lambda x: (x["bbox"]["y"], x["bbox"]["x"])):
        b = r["bbox"]
        overlap = False
        for k in kept:
            kb = k["bbox"]
            xo = max(0, min(b["x"]+b["width"], kb["x"]+kb["width"]) - max(b["x"], kb["x"]))
            yo = max(0, min(b["y"]+b["height"], kb["y"]+kb["height"]) - max(b["y"], kb["y"]))
            if xo > 0 and yo > 0 and (xo*yo)/min(b["width"]*b["height"], kb["width"]*kb["height"]) > 0.5:
                overlap = True; break
        if not overlap:
            kept.append(r)
    return kept

def resegment_region(img, parent, img_h, img_w):
    region = parent["crop"]
    rh, rw = region.shape[:2]
    gray = cv2.cvtColor(region, cv2.COLOR_RGB2GRAY)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (rw // 3, max(3, rh // 120)))
    dilated = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

    proj = np.sum(dilated, axis=1) // 255
    thresh = rw * 0.01
    content = proj > thresh

    bands = []
    in_b = False
    start = 0
    min_h = max(15, rh // 80)
    for i in range(len(content)):
        if content[i] and not in_b:
            start = i; in_b = True
        elif not content[i] and in_b:
            if i - start > min_h:
                bands.append((start, i))
            in_b = False
    if in_b and len(content) - start > min_h:
        bands.append((start, len(content)))

    px, py = parent["bbox"]["x"], parent["bbox"]["y"]
    results = []
    for ry1, ry2 in bands:
        x1 = max(0, px - 2)
        x2 = min(img_w, px + rw + 2)
        y1 = max(0, py + ry1 - 3)
        y2 = min(img_h, py + ry2 + 3)
        crop = img[y1:y2, x1:x2]
        if crop.shape[0] >= 20 and crop.shape[1] >= 20:
            results.append({"idx": 0, "bbox": {"x": int(x1), "y": y1, "width": int(x2 - x1), "height": int(y2 - y1)}, "crop": crop})
    return results or [parent]

def segment_questions(img):
    h, w = img.shape[:2]
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

    k_h = max(5, h // 80)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (w // 4, k_h))
    dilated = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

    row_proj = np.sum(dilated, axis=1) // 255
    threshold = w * 0.015
    content_rows = row_proj > threshold

    row_bands = []
    in_band = False
    start = 0
    min_h = max(20, h // 60)
    for i in range(len(content_rows)):
        if content_rows[i] and not in_band:
            start = i; in_band = True
        elif not content_rows[i] and in_band:
            if i - start > min_h:
                row_bands.append((start, i))
            in_band = False
    if in_band and len(content_rows) - start > min_h:
        row_bands.append((start, len(content_rows)))

    questions = []
    q_idx = 0

    for ry1, ry2 in row_bands:
        row_img = binary[ry1:ry2, :]
        rh = row_img.shape[0]
        vkw = max(3, w // 60)
        vkh = max(3, int(rh * 0.25))
        vk = cv2.getStructuringElement(cv2.MORPH_RECT, (vkw, vkh))
        rd = cv2.morphologyEx(row_img, cv2.MORPH_CLOSE, vk)

        col_proj = np.sum(rd, axis=0) // 255
        ct = rh * 0.05

        col_bands = []
        in_b = False
        s = 0
        mcw = max(20, w // 40)
        for j in range(len(col_proj)):
            if col_proj[j] > ct and not in_b:
                s = j; in_b = True
            elif col_proj[j] <= ct and in_b:
                if j - s > mcw:
                    col_bands.append((s, j))
                in_b = False
        if in_b and len(col_proj) - s > mcw:
            col_bands.append((s, len(col_proj)))

        if not col_bands:
            col_bands = [(0, w)]
        elif len(col_bands) > 1:
            wide = [b for b in col_bands if (b[1]-b[0]) >= w*0.2]
            if 1 <= len(wide) <= 2:
                col_bands = wide
            else:
                valid = sum(b[1]-b[0] for b in col_bands)
                if valid < w * 0.15:
                    col_bands = [(0, w)]

        for cx1, cx2 in col_bands:
            x1 = max(0, cx1 - 4)
            x2 = min(w, cx2 + 4)
            y1 = max(0, ry1 - 4)
            y2 = min(h, ry2 + 4)
            crop = img[y1:y2, x1:x2]
            if crop.shape[0] < 25 or crop.shape[1] < 25:
                continue
            q_idx += 1
            questions.append({"idx": q_idx, "bbox": {"x": int(x1), "y": int(y1), "width": int(x2-x1), "height": int(y2-y1)}, "crop": crop})

    if questions:
        questions = filter_question_regions(questions, h, w)
        refined = []
        for q in questions:
            if q["bbox"]["height"] > 350:
                refined.extend(resegment_region(img, q, h, w))
            else:
                refined.append(q)
        refined = merge_fragments(refined)
        refined = deduplicate_overlaps(refined)
        for ni, q in enumerate(refined, 1):
            q["idx"] = ni
        questions = refined
    return questions

# --- JSON extraction ---
def extract_json(text):
    cleaned = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
    cleaned = re.sub(r'```json\s*|```\s*', '', cleaned).strip()
    start = cleaned.find('{')
    if start == -1:
        return None, "no JSON"
    depth = 0
    end = -1
    for i in range(start, len(cleaned)):
        if cleaned[i] == '{':
            depth += 1
        elif cleaned[i] == '}':
            depth -= 1
        if depth == 0:
            end = i + 1
            break
    if end == -1:
        return None, "unmatched braces"
    cleaned = cleaned[start:end]
    try:
        return json.loads(cleaned), None
    except json.JSONDecodeError:
        cleaned = re.sub(r',\s*}', '}', cleaned)
        cleaned = re.sub(r',\s*]', ']', cleaned)
        try:
            return json.loads(cleaned), None
        except json.JSONDecodeError:
            pass
    return None, "parse failed"

# --- Per-question analysis prompt (compact JSON schema format) ---
PER_QUESTION_PROMPT = (
    'You are analyzing a SINGLE QUESTION crop from a children\'s worksheet.\n'
    'Read the image CAREFULLY. Your MOST IMPORTANT task: extract ALL text EXACTLY as written.\n\n'
    'Return ONLY valid JSON with this exact structure (no markdown, no thinking):\n'
    '{\n'
    '  "question_text": "EXACT instruction text verbatim. Read every word. If no text, set empty string.",\n'
    '  "all_visible_text": [{"text": "...", "position": "top|middle|bottom|left|right|center|label"}],\n'
    '  "instruction_verb": "Count|Match|Circle|Colour|Write|Trace|Tick|Draw|Join|Add|Subtract|Fill|Solve|Find|Complete",\n'
    '  "question_type": "Counting|Matching|Addition|Subtraction|Number-Recognition|Pattern|Comparison|Shape-Recognition|Fill-Blank|Sequencing|Tracing|Writing-Practice",\n'
    '  "question_sub_type": "Count-And-Write|Circle-Correct|Match-Column|Fill-Missing|Greater-Less|Before-After-Between|Ascending|Descending|True-False|Story-Sum|Complete-Pattern",\n'
    '  "structure": "single-item|multiple-items|matching-columns|grid|with-illustration|text-only|horizontal-row|vertical-list",\n'
    '  "item_count": 0,\n'
    '  "items": [{"item_number": 1, "text": "item label", "objects": [], "answer_space": {}}],\n'
    '  "objects": [{"name": "object name (singular)", "count": 5, "attributes": ["red"], "position": "center|scattered|row", "arrangement": "row|grid|scattered|grouped"}],\n'
    '  "total_object_count": 0,\n'
    '  "illustration_purpose": "counting|identification|matching|comparison|story-context|pattern-recognition|number-recognition",\n'
    '  "answer_spaces": [{"type": "blank|box|circle|line|dotted-line|tick-box|bracket", "count": 1, "location": "below-text|beside-text|inside-text|at-end", "associated_item_number": 0}],\n'
    '  "visual_elements": ["border|dotted-line|arrow|number-label|frame|underline|box-border"],\n'
    '  "colors_mentioned": ["red|blue|green|yellow|orange|purple|pink|brown|black"],\n'
    '  "relational_words": ["more|less|bigger|smaller|same|different|before|after|between|greater|fewer"],\n'
    '  "group_context": {"is_grouped": false, "group_id": 0, "group_instruction": ""},\n'
    '  "concept": "FLN concept name",\n'
    '  "learning_outcome": "measurable skill",\n'
    '  "cognitive_skill": "Remembering|Understanding|Applying|Analyzing",\n'
    '  "motor_skill": "Writing|Circling|Matching|Colouring|Tracing|Drawing",\n'
    '  "difficulty": "Easy|Medium|Hard"\n'
    '}\n\n'
    'RULES: question_text must be WORD FOR WORD exact copy. all_visible_text includes EVERY number, label, word fragment. '
    'List each distinct object type once with precise count. '
    'No answer/expected_answer/correct_answer field. '
    'Use "" for empty text, [] for empty lists, 0 for zero, false for booleans.'
)

def analyze_question_crop(llm, crop_img, name):
    hc, wc = crop_img.shape[:2]
    if min(hc, wc) > 400:
        est = (hc // 16) * (wc // 16)
        if est > 3000:
            print(f"\n    NOTE: Large crop ({wc}x{hc}, ~{est} img tokens)")
    _, buffer = cv2.imencode(".png", cv2.cvtColor(crop_img, cv2.COLOR_RGB2BGR))
    b64 = base64.b64encode(buffer).decode("utf-8")
    result_data = {"file": name, "raw": "", "parsed": None, "error": "max retries", "attempts": 3}
    for attempt in range(1, 4):
        hint = ["", "IMPORTANT: Return valid JSON with ALL fields. Do NOT include answer.",
                "CRITICAL: Valid JSON only. Use \"\" for text, [] for lists."][attempt-1]
        text = PER_QUESTION_PROMPT
        if hint:
            text += "\n\n" + hint
        resp = llm.create_chat_completion(
            messages=[{"role": "user", "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                {"type": "text", "text": text + "\n\nExtract everything visible in this question crop."},
            ]}],
            max_tokens=6144, temperature=0.1 + (attempt-1)*0.1,
        )
        raw = resp["choices"][0]["message"]["content"]
        parsed, error = extract_json(raw)
        if parsed and isinstance(parsed, dict):
            qt = str(parsed.get("question_text", "") or "").strip()
            qtype = str(parsed.get("question_type", "") or "").strip()
            if qt or qtype or parsed.get("items", []):
                result_data = {"file": name, "raw": raw, "parsed": parsed, "error": None, "attempts": attempt}
                break
            result_data = {"file": name, "raw": raw, "parsed": parsed, "error": "all fields empty", "attempts": attempt}
        else:
            result_data = {"file": name, "raw": raw, "parsed": None, "error": error or "parse failed", "attempts": attempt}
    return result_data

print("Segmentation + extraction functions loaded")


In [ ]:
# Cell 6: Find images → Segment → Analyze each crop → Collect per-question JSON

ZEXT = "/kaggle/working/zip_extract"
if os.path.exists(ZEXT):
    shutil.rmtree(ZEXT)
os.makedirs(ZEXT, exist_ok=True)

image_paths = []
for root, _, files in os.walk("/kaggle/input"):
    for fn in sorted(files):
        fp = os.path.join(root, fn)
        if fn.lower().endswith(".zip"):
            with zipfile.ZipFile(fp) as z:
                z.extractall(ZEXT)
            print(f"  Extracted: {fn}")
        elif fn.lower().endswith((".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tiff")):
            image_paths.append(fp)

for root, _, files in os.walk(ZEXT):
    for fn in sorted(files):
        if fn.lower().endswith((".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tiff")):
            image_paths.append(os.path.join(root, fn))

if not image_paths:
    print("No images found in /kaggle/input.")
    print("Upload images using the Add Data button (top right), then re-run this cell.")
else:
    print(f"Found {len(image_paths)} image(s)")

all_worksheets = []

for idx, img_path in enumerate(image_paths, 1):
    name = Path(img_path).name
    stem = Path(name).stem
    img_dir = os.path.join(OUTPUT_DIR, stem)
    os.makedirs(img_dir, exist_ok=True)
    crops_dir = os.path.join(img_dir, "crops")
    os.makedirs(crops_dir, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"[{idx}/{len(image_paths)}] {name}")
    print(f"{'='*60}")

    img = cv2.imread(img_path)
    if img is None:
        print(f"  FAILED: cannot read {name}")
        continue
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    print(f"  Input: {img.shape[1]}x{img.shape[0]}")

    img, info = preprocess(img)
    if info["applied"]:
        print(f"  Preprocessing: {', '.join(info['applied'])}")
    else:
        print("  Preprocessing: none needed")

    enhanced_path = os.path.join(img_dir, name)
    cv2.imwrite(enhanced_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
    display(IPyImage(img_path))

    # --- SEGMENT into individual questions ---
    questions = segment_questions(img)
    for q in questions:
        b = q["bbox"]
        q["crop"] = img[b["y"]:b["y"]+b["height"], b["x"]:b["x"]+b["width"]]
    print(f"  Questions detected: {len(questions)}")

    # --- Save individual crop images ---
    for q in questions:
        crop_path = os.path.join(crops_dir, f"q{q['idx']:02d}.png")
        cv2.imwrite(crop_path, cv2.cvtColor(q["crop"], cv2.COLOR_RGB2BGR))

    # --- ANALYZE each question crop with Gemma ---
    worksheet_data = {
        "worksheet": name,
        "worksheet_info": {"width": int(img.shape[1]), "height": int(img.shape[0]), "preprocessing": info},
        "questions": []
    }

    _t_start = time.time()
    for q in questions:
        q_idx = q["idx"]
        crop_name = f"q{q_idx:02d}.png"
        print(f"  [{q_idx}/{len(questions)}] Analyzing...", end=" ", flush=True)

        result = analyze_question_crop(llm, q["crop"], crop_name)
        spec = result.get("parsed") or {}
        confidence = max(0.0, min(1.0, 1.0 - (result["attempts"] - 1) * 0.1)) if result["parsed"] else 0.0

        entry = {
            "question_number": q_idx,
            "bounding_box": q["bbox"],
            "analysis_attempts": result["attempts"],
            "analysis_error": result["error"],
            "confidence": round(confidence, 2),
            "raw_response": result.get("raw", "") or "",
            "question_text": spec.get("question_text", "") or "",
            "all_visible_text": spec.get("all_visible_text", []) or [],
            "instruction_verb": spec.get("instruction_verb", "") or "",
            "question_type": spec.get("question_type", "") or "",
            "question_sub_type": spec.get("question_sub_type", "") or "",
            "structure": spec.get("structure", "") or "",
            "item_count": spec.get("item_count", 0) or 0,
            "items": spec.get("items", []) or [],
            "objects": spec.get("objects", []) or [],
            "total_object_count": spec.get("total_object_count", 0) or 0,
            "illustration_purpose": spec.get("illustration_purpose", "") or "",
            "answer_spaces": spec.get("answer_spaces", []) or [],
            "visual_elements": spec.get("visual_elements", []) or [],
            "colors_mentioned": spec.get("colors_mentioned", []) or [],
            "relational_words": spec.get("relational_words", []) or [],
            "group_context": spec.get("group_context", {}) or {},
            "concept": spec.get("concept", "") or "",
            "learning_outcome": spec.get("learning_outcome", "") or "",
            "cognitive_skill": spec.get("cognitive_skill", "") or "",
            "motor_skill": spec.get("motor_skill", "") or "",
            "difficulty": spec.get("difficulty", "") or "",
        }
        worksheet_data["questions"].append(entry)

        qt_preview = entry["question_text"][:60] if entry["question_text"] else "[empty]"
        qt_type = entry["question_type"] or "[unknown]"
        print(f"Q{q_idx}: {qt_type} | {qt_preview}")

    elapsed = time.time() - _t_start
    print(f"  Elapsed: {elapsed:.0f}s ({elapsed/60:.1f}min)")

    # --- Save per-worksheet JSON ---
    out_path = os.path.join(img_dir, f"{stem}_questions.json")
    with open(out_path, "w") as f:
        json.dump(worksheet_data, f, indent=2, ensure_ascii=False)
    print(f"  Saved: {out_path}")
    all_worksheets.append(worksheet_data)

# --- Summary ---
if image_paths:
    total_qs = sum(len(w["questions"]) for w in all_worksheets)
    print(f"\n{'='*60}")
    print(f"COMPLETE: {len(all_worksheets)} worksheet(s), {total_qs} questions extracted")
    print(f"{'='*60}")
    manifest_path = os.path.join(OUTPUT_DIR, "batch_manifest.json")
    with open(manifest_path, "w") as f:
        json.dump({"worksheets": all_worksheets}, f, indent=2, ensure_ascii=False)
    print(f"Manifest: {manifest_path}")
    # --- Batch CSV summary ---
    import csv
    csv_summary = os.path.join(OUTPUT_DIR, "all_questions_summary.csv")
    with open(csv_summary, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["worksheet","qnum","type","subtype","verb","text","objects","concept","difficulty","confidence"])
        for ws in all_worksheets:
            fn = ws["worksheet"]
            for qe in ws["questions"]:
                w.writerow([fn, qe["question_number"], qe["question_type"], qe["question_sub_type"],
                            qe["instruction_verb"], qe["question_text"][:200], qe["total_object_count"],
                            qe["concept"], qe["difficulty"], qe["confidence"]])
    print(f"CSV summary: {csv_summary}")


In [ ]:
# Cell 7: Package results

!zip -r /kaggle/working/FLN_Results.zip /kaggle/working/FLN_Results
print("Done. Download FLN_Results.zip from the Output tab (top right).")
